# Phase 2 — Preprocessing & Feature Engineering

**Objective:** Clean text data, apply NLP preprocessing (tokenization, stop-word removal, stemming), and engineer both text-based (TF-IDF) and structural metadata features.

## Phase 2 Checklist
- [ ] Load combined dataset from Phase 1
- [ ] Text cleaning (lowercase, remove punctuation/numbers)
- [ ] Tokenization
- [ ] Stop-word removal
- [ ] Stemming
- [ ] Build full preprocessing pipeline
- [ ] TF-IDF vectorization
- [ ] Combine TF-IDF + structural features
- [ ] Train/test split
- [ ] Save processed data for Phase 3


In [1]:
import pandas as pd
import numpy as np
import re
import string
import nltk
import joblib
import os

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix

import warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('All imports successful')

All imports successful


## 📂 Step 1 — Load Combined Dataset from Phase 1

In [2]:
df = pd.read_csv('../data/raw/combined_dataset.csv')
print(f'Shape: {df.shape}')
print(df['label'].value_counts())
df.head()

Shape: (55, 2)
label
ham     30
spam    25
Name: count, dtype: int64


,label,message
0,spam,Your free ringtone is waiting! Text MIX to 850...
1,ham,"Just landed, will call you when I get my bags"
2,spam,Camera - you asked to be in our database? Txt ...
3,ham,Hey. What time does the movie start?
4,spam,FREE iPhone if you complete our survey. Click ...


## 🧹 Step 2 — Text Cleaning Function

We'll clean each message by:
- Lowercasing
- Removing URLs, email addresses
- Removing punctuation & numbers
- Removing extra whitespace

In [3]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)   # remove URLs
    text = re.sub(r'\S+@\S+', '', text)                     # remove emails
    text = re.sub(r'[^a-z\s]', ' ', text)                   # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()                # remove extra whitespace
    return text

df['clean_text'] = df['message'].apply(clean_text)
df[['message', 'clean_text']].head(10)

,message,clean_text
0,Your free ringtone is waiting! Text MIX to 850...,your free ringtone is waiting text mix to to v...
1,"Just landed, will call you when I get my bags",just landed will call you when i get my bags
2,Camera - you asked to be in our database? Txt ...,camera you asked to be in our database txt yes...
3,Hey. What time does the movie start?,hey what time does the movie start
4,FREE iPhone if you complete our survey. Click ...,free iphone if you complete our survey click h...
5,Subject: You Are A Winner\nCongratulations! Cl...,subject you are a winner congratulations claim...
6,Subject: Investment Opportunity\nDouble your s...,subject investment opportunity double your sav...
7,Congratulations! You have won 1000 cash or a t...,congratulations you have won cash or a top son...
8,"Subject: Meeting Tomorrow\nHi team, reminder t...",subject meeting tomorrow hi team reminder that...
9,"Running late, be there in 20 mins sorry",running late be there in mins sorry


## ✂️ Step 3 — Tokenization, Stop-word Removal & Stemming

In [4]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words and len(word) > 2]
    return ' '.join(tokens)

df['processed_text'] = df['clean_text'].apply(preprocess_text)
df[['message', 'clean_text', 'processed_text']].head(10)

,message,clean_text,processed_text
0,Your free ringtone is waiting! Text MIX to 850...,your free ringtone is waiting text mix to to v...,free rington wait text mix verifi get usher ri...
1,"Just landed, will call you when I get my bags",just landed will call you when i get my bags,land call get bag
2,Camera - you asked to be in our database? Txt ...,camera you asked to be in our database txt yes...,camera ask databas txt ye get voucher code wor...
3,Hey. What time does the movie start?,hey what time does the movie start,hey time movi start
4,FREE iPhone if you complete our survey. Click ...,free iphone if you complete our survey click h...,free iphon complet survey click claim
5,Subject: You Are A Winner\nCongratulations! Cl...,subject you are a winner congratulations claim...,subject winner congratul claim reward call free
6,Subject: Investment Opportunity\nDouble your s...,subject investment opportunity double your sav...,subject invest opportun doubl save guarante li...
7,Congratulations! You have won 1000 cash or a t...,congratulations you have won cash or a top son...,congratul cash top soni call
8,"Subject: Meeting Tomorrow\nHi team, reminder t...",subject meeting tomorrow hi team reminder that...,subject meet tomorrow team remind weekli sync ...
9,"Running late, be there in 20 mins sorry",running late be there in mins sorry,run late min sorri


## 📊 Step 4 — Structural Metadata Features

Per the assignment: email length, hyperlinks, special characters, exclamation marks, spam trigger words, HTML vs plain text ratio.

In [5]:
spam_trigger_words = ['free', 'win', 'winner', 'cash', 'prize', 'urgent', 'claim',
                       'congratulations', 'click', 'offer', 'guarantee', 'limited',
                       'act now', 'call now', 'credit', 'loan', 'discount']

def extract_structural_features(df):
    df = df.copy()
    df['msg_length']    = df['message'].str.len()
    df['word_count']    = df['message'].str.split().str.len()
    df['num_links']     = df['message'].str.count(r'http[s]?://|www\.')
    df['num_exclaim']   = df['message'].str.count(r'!')
    df['num_special']   = df['message'].str.count(r'[^a-zA-Z0-9\s]')
    df['num_digits']    = df['message'].str.count(r'\d')
    df['num_upper']     = df['message'].str.count(r'[A-Z]')
    df['upper_ratio']   = df['num_upper'] / (df['msg_length'] + 1)
    df['has_html']      = df['message'].str.contains(r'<[^>]+>', regex=True).astype(int)
    df['trigger_count'] = df['message'].str.lower().apply(
        lambda x: sum(1 for word in spam_trigger_words if word in x)
    )
    return df

df = extract_structural_features(df)
structural_cols = ['msg_length','word_count','num_links','num_exclaim',
                    'num_special','num_digits','upper_ratio','has_html','trigger_count']
df[structural_cols].describe()

,msg_length,word_count,num_links,num_exclaim,num_special,num_digits,upper_ratio,has_html,trigger_count
count,55.000000,55.000000,55.0,55.000000,55.000000,55.000000,55.000000,55.0,55.000000
mean,73.690909,13.327273,0.0,0.418182,2.181818,2.672727,0.075062,0.0,1.018182
std,24.456214,3.949001,0.0,0.685590,1.263313,4.422395,0.097023,0.0,1.521341
min,29.000000,6.000000,0.0,0.000000,0.000000,0.000000,0.016949,0.0,0.000000
25%,50.500000,10.500000,0.0,0.000000,1.000000,0.000000,0.037037,0.0,0.000000
50%,83.000000,14.000000,0.0,0.000000,2.000000,0.000000,0.054945,0.0,0.000000
75%,95.000000,16.000000,0.0,1.000000,3.000000,3.000000,0.080032,0.0,2.000000
max,108.000000,23.000000,0.0,3.000000,6.000000,18.000000,0.722222,0.0,5.000000


## 🔢 Step 5 — TF-IDF Vectorization

Converting cleaned text into numerical feature vectors using TF-IDF (max 3000 features, including unigrams + bigrams).

In [6]:
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2), min_df=2)
X_tfidf = tfidf.fit_transform(df['processed_text'])

print(f'TF-IDF matrix shape: {X_tfidf.shape}')
print(f'Sample features: {tfidf.get_feature_names_out()[:20]}')

TF-IDF matrix shape: (55, 83)
Sample features: ['ask' 'birthday' 'call' 'call claim' 'call get' 'camera' 'card' 'cash'
 'chanc' 'claim' 'claim prize' 'click' 'click claim' 'code' 'come'
 'congratul' 'day' 'dear' 'dinner' 'earn']


## 🔗 Step 6 — Combine TF-IDF + Structural Features

Scale the structural features and stack them with the sparse TF-IDF matrix.

In [7]:
scaler = StandardScaler()
X_structural = scaler.fit_transform(df[structural_cols])
X_structural_sparse = csr_matrix(X_structural)

X_combined = hstack([X_tfidf, X_structural_sparse])
print(f'Combined feature matrix shape: {X_combined.shape}')

y = df['label'].map({'ham': 0, 'spam': 1})
print(f'\nLabel distribution:\n{y.value_counts()}')

Combined feature matrix shape: (55, 92)

Label distribution:
label
0    30
1    25
Name: count, dtype: int64


## ✂️ Step 7 — Train/Test Split

80/20 split, stratified to preserve the spam/ham ratio in both sets.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train set: {X_train.shape[0]} samples')
print(f'Test set:  {X_test.shape[0]} samples')
print(f'\nTrain label distribution:\n{y_train.value_counts()}')
print(f'\nTest label distribution:\n{y_test.value_counts()}')

Train set: 44 samples
Test set:  11 samples

Train label distribution:
label
0    24
1    20
Name: count, dtype: int64

Test label distribution:
label
0    6
1    5
Name: count, dtype: int64


## 💾 Step 8 — Save Everything for Phase 3

In [10]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

joblib.dump(X_train, '../data/processed/X_train.pkl')
joblib.dump(X_test, '../data/processed/X_test.pkl')
joblib.dump(y_train, '../data/processed/y_train.pkl')
joblib.dump(y_test, '../data/processed/y_test.pkl')

joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')
joblib.dump(scaler, '../models/feature_scaler.pkl')

df.to_csv('../data/processed/processed_dataset.csv', index=False)

print('Saved:')
print('   data/processed/X_train.pkl, X_test.pkl, y_train.pkl, y_test.pkl')
print('   models/tfidf_vectorizer.pkl, feature_scaler.pkl')
print('   data/processed/processed_dataset.csv')

Saved:
   data/processed/X_train.pkl, X_test.pkl, y_train.pkl, y_test.pkl
   models/tfidf_vectorizer.pkl, feature_scaler.pkl
   data/processed/processed_dataset.csv


## Phase 2 Complete!

| Step | Task | Status |
|------|------|--------|
| 1 | Load combined dataset | ✅ |
| 2 | Text cleaning | ✅ |
| 3 | Tokenization, stop-words, stemming | ✅ |
| 4 | Structural metadata features | ✅ |
| 5 | TF-IDF vectorization | ✅ |
| 6 | Combine text + structural features | ✅ |
| 7 | Train/test split | ✅ |
| 8 | Save artifacts for Phase 3 | ✅ |

▶️ **Next: Phase 3 — Train all 5 classifiers (Naive Bayes, Logistic Regression, Random Forest, SVM, LSTM)**